# Benchmark an LLM on real devices with TinyEdge

[TinyEdge](https://tinyedge.ai) runs GGUF models on real phones and tablets and reports decode tok/s, time-to-first-token, RAM, and perplexity. This notebook:

1. Benchmarks a model on your devices.
2. Runs `optimize=True` to compare quantizations and see which one to ship per device.

**Before you run:** a free [tinyedge.ai](https://tinyedge.ai) account (comes with demo credit; this notebook costs about a dollar), your API key in the first cell, at least one device paired in the TinyEdge Runner app with "Available for benchmarks" on, and internet enabled (Kaggle: right sidebar).

In [ ]:
%pip -q install "tinyedge[hf]>=0.3.0"

In [ ]:
import tinyedge

# Paste your API key from tinyedge.ai (New benchmark > Your API key):
client = tinyedge.TinyEdge(api_key="tinyedge_sk_REPLACE_ME")

DEVICES = client.devices(online=True)   # devices online right now
print("benchmarking on:", DEVICES)

## Benchmark a model

Pass an `hf:<owner>/<repo>/<file.gguf>` reference, or a local path. Swap in any GGUF (Qwen2.5, Gemma-2, Phi-3, your own).

In [ ]:
MODEL = "hf:bartowski/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct-Q4_K_M.gguf"

# small WikiText sample so the run also measures perplexity
import io, zipfile, pathlib, requests
pathlib.Path("corpus").mkdir(exist_ok=True)
wiki = zipfile.ZipFile(io.BytesIO(requests.get(
    "https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip", timeout=120).content))
pathlib.Path("corpus/wiki.txt").write_bytes(wiki.read("wikitext-2-raw/wiki.test.raw")[:200_000])

In [ ]:
# one row per device: tok/s, time-to-first-token, RAM, perplexity
client.benchmark(MODEL, devices=DEVICES, dataset="corpus")

## Compare quantizations (optimize=True)

`optimize=True` builds the quant ladder from a local f16, drops variants that hurt quality, and benchmarks them as one sweep, reusing the corpus for perplexity. A small model keeps it quick.

In [ ]:
from huggingface_hub import hf_hub_download
F16 = hf_hub_download("bartowski/SmolLM2-135M-Instruct-GGUF", "SmolLM2-135M-Instruct-f16.gguf")

report = client.benchmark(F16, devices=DEVICES, dataset="corpus", optimize=True)
print(report.summary())
print("full report:", report.sweep_url)